Celda 1 — Imports y rutas

In [1]:
import os
import numpy as np
import pandas as pd

# Ajusta esta ruta a tu carpeta del PASO 0
BASE_DIR = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0"

# Archivo Excel fuente (etiquetado manual)
XLSX_PATH = os.path.join(BASE_DIR, "ETIQUETADO DE DATOS.xlsx")

# Salidas
OUT_DATASET = os.path.join(BASE_DIR, "dataset_v1_etiquetado_v2.csv")
OUT_REVIEW  = os.path.join(BASE_DIR, "review_ok_rows_with_labels.csv")

print("BASE_DIR:", BASE_DIR)
print("XLSX_PATH:", XLSX_PATH)

BASE_DIR: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0
XLSX_PATH: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0\ETIQUETADO DE DATOS.xlsx


Celda 2 — Cargar el Excel

In [2]:
df = pd.read_excel(XLSX_PATH)

print("Filas:", df.shape[0], "Columnas:", df.shape[1])
df.head()

Filas: 793 Columnas: 29


,clip_id,file_name,start_time_s,end_time_s,duration_s,audio_layout,program_type,segment_type,lufs_integrated,true_peak_dbfs,...,label_distortion,source_format,source_origin,pre_correction,qc_result,annotator,synthetic_case,synthetic_type,confidence,notes
0,13301-LATIDOS-CORAPLUS-GIRASOl_1,13301-LATIDOS-CORAPLUS-GIRASOL.wav,0,5,5,STEREO,COMERCIAL,MIXTO,NaN,NaN,...,0,WAV,CANAL,0,OK,AA,0,0,5,NaN
1,13301-LATIDOS-CORAPLUS-GIRASOl_2,13301-LATIDOS-CORAPLUS-GIRASOL.wav,5,10,5,STEREO,COMERCIAL,MIXTO,NaN,NaN,...,0,WAV,CANAL,0,OK,AA,0,0,5,NaN
2,13301-LATIDOS-CORAPLUS-GIRASOl_3,13301-LATIDOS-CORAPLUS-GIRASOL.wav,10,15,5,STEREO,COMERCIAL,MIXTO,NaN,NaN,...,0,WAV,CANAL,0,OK,AA,0,0,5,0
3,13301-LATIDOS-CORAPLUS-GIRASOl_4,13301-LATIDOS-CORAPLUS-GIRASOL.wav,15,20,5,STEREO,COMERCIAL,MIXTO,NaN,NaN,...,0,WAV,CANAL,0,OK,AA,0,0,5,0
4,13301-LATIDOS-CORAPLUS-GIRASOl_5,13301-LATIDOS-CORAPLUS-GIRASOL.wav,20,25,5,STEREO,COMERCIAL,MIXTO,NaN,NaN,...,0,WAV,CANAL,0,OK,AA,0,0,5,0


Celda 3 — Validación mínima de columnas

In [3]:
required_cols = ["file_name", "start_time_s", "end_time_s", "qc_result"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Faltan columnas requeridas en el Excel: {missing}")

label_cols = [c for c in df.columns if c.startswith("label_")]
if len(label_cols) == 0:
    raise ValueError("No se encontraron columnas label_*. Revisa el Excel.")

print("Labels detectadas:", label_cols)

Labels detectadas: ['label_loudness_high', 'label_loudness_low', 'label_clipping', 'label_noise_high', 'label_silence_anomalous', 'label_mono_issue', 'label_distortion']


Celda 4 — Normalizar qc_result (OK / advertencia / falla)

In [4]:
def norm_qc(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if not s:
        return np.nan
    s_low = s.lower()

    if s_low in ["ok", "okay", "pass"]:
        return "OK"
    if s_low in ["advertencia", "warning", "warn"]:
        return "advertencia"
    if s_low in ["falla", "fail", "error"]:
        return "falla"

    # Si alguien escribió raro, se conserva pero lo veremos en conteos
    return s

df["qc_result"] = df["qc_result"].apply(norm_qc)

print(df["qc_result"].value_counts(dropna=False))

qc_result
OK             435
falla          348
advertencia     10
Name: count, dtype: int64


Celda 5 — Convertir labels a 0/1 (y detectar valores raros)

In [5]:
# Convertimos a numérico
for c in label_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Revisión de valores no 0/1 (solo para alertar)
bad_mask = df[label_cols].notna() & (~df[label_cols].isin([0, 1]))
bad_cells = int(bad_mask.sum().sum())
bad_rows = int(bad_mask.any(axis=1).sum())

print("Celdas con valores no 0/1:", bad_cells)
print("Filas afectadas:", bad_rows)

# Si quieres forzar a 0/1 (por ejemplo si hay True/False), podrías mapear,
# pero lo ideal es que el Excel ya tenga 0/1.

Celdas con valores no 0/1: 0
Filas afectadas: 0


Celda 6 — Filtrar filas etiquetadas (qc_result no vacío)

In [6]:
df_labeled = df[df["qc_result"].notna()].copy()

print("Filas con qc_result definido:", len(df_labeled))

Filas con qc_result definido: 793


Celda 7 — Regla de consistencia: OK con labels activas → advertencia

In [7]:
# Suma de labels por fila (NaN se trata como 0)
label_sum = df_labeled[label_cols].fillna(0).sum(axis=1)

mask_ok_with_labels = (df_labeled["qc_result"] == "OK") & (label_sum > 0)

print("Filas OK con alguna label activa:", int(mask_ok_with_labels.sum()))

# Guardamos estas filas para revisión (opcional)
df_review = df_labeled.loc[mask_ok_with_labels, ["file_name","start_time_s","end_time_s","qc_result"] + label_cols].copy()

# Aplicamos la regla acordada
df_labeled.loc[mask_ok_with_labels, "qc_result"] = "advertencia"

Filas OK con alguna label activa: 7


Celda 8 — Crear y_bin (OK vs NO_OK)

In [8]:
df_labeled["y_bin"] = df_labeled["qc_result"].apply(lambda s: 0 if s == "OK" else 1)

print("Distribución qc_result:")
print(df_labeled["qc_result"].value_counts())

print("\nDistribución y_bin:")
print(df_labeled["y_bin"].value_counts())

Distribución qc_result:
qc_result
OK             428
falla          348
advertencia     17
Name: count, dtype: int64

Distribución y_bin:
y_bin
0    428
1    365
Name: count, dtype: int64


Celda 9 — Ordenar y exportar CSV oficial (dataset_v1_etiquetado_v2)

In [9]:
# Orden reproducible (si existen las columnas)
sort_cols = [c for c in ["file_name", "start_time_s", "end_time_s"] if c in df_labeled.columns]
if sort_cols:
    df_labeled = df_labeled.sort_values(sort_cols).reset_index(drop=True)

# Exportar dataset final
os.makedirs(BASE_DIR, exist_ok=True)
df_labeled.to_csv(OUT_DATASET, index=False, encoding="utf-8")

# Exportar revisión opcional
df_review.to_csv(OUT_REVIEW, index=False, encoding="utf-8")

print("✅ Dataset exportado:", OUT_DATASET)
print("✅ Filas para revisión (opcional):", OUT_REVIEW)
print("Total filas exportadas:", len(df_labeled))

✅ Dataset exportado: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0\dataset_v1_etiquetado_v2.csv
✅ Filas para revisión (opcional): C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0\review_ok_rows_with_labels.csv
Total filas exportadas: 793


Celda 10 — Checklist rápido final (sanity check)

In [10]:
# 1) No NaN en qc_result
assert df_labeled["qc_result"].isna().sum() == 0

# 2) y_bin solo 0/1
assert set(df_labeled["y_bin"].unique()).issubset({0,1})

# 3) Labels válidas (0/1 o NaN si alguna quedó vacía)
invalid = df_labeled[label_cols].notna() & (~df_labeled[label_cols].isin([0,1]))
print("Labels inválidas (debería ser 0):", int(invalid.sum().sum()))

print("✅ Sanity check completado.")

Labels inválidas (debería ser 0): 0
✅ Sanity check completado.
